## DKL Pricer

### Simulating GBM (as described in Chapter 4 in the report)

In [ ]:
!pip install gpytorch
import gpytorch
import numpy as np
import torch
import torch.nn as nn
import time

np.random.seed(7)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

#Simulating GBM (as described in Chapter 4 in the report)
def simulate_gbm(S0, d, N, K, r, q, sigma, rho, T, n_dates,
                 device = device, dtype = dtype):

    dt = T/n_dates
    
    S = torch.empty((N,n_dates + 1, d), device = device, dtype = dtype)
    S[:,0,:] = S0

    pi = torch.full((d,d), rho, device = device, dtype = dtype)
    pi.fill_diagonal_(1.0)
    L = torch.linalg.cholesky(pi)                   #cholesky is lower triang decomposition where pi = L*L^T

    for i in range(n_dates):
        z = torch.randn((N,d), device = device, dtype = dtype)
        omega = z @ L.T                              #var(z*L^T) = Lvar(z)LT = L*I*L^T = L*L^T = var(omega)
        S[:,i+1,:] =  S[:,i,:]*torch.exp((r-q-0.5*sigma**2)*dt + sigma*torch.sqrt(torch.tensor(dt, device = device, dtype = dtype))*omega)

    return S


def payoff_max_call(S_t, K):
    return torch.relu(S_t.max(dim = -1).values - K) 

### DKL model and the backward recursion algorithm (Algorithm 1 in Chapter 1)

In [ ]:
#feature extractor

class FNNfeatureExt(nn.Module):
    def __init__(self, d_input):
        super().__init__()
        self.feature_ext = nn.Sequential(
            torch.nn.Linear(d_input, 1000),
            torch.nn.ReLU(),
            torch.nn.Linear(1000, 500),
            torch.nn.ReLU(),
            torch.nn.Linear(500, 50),
            torch.nn.ReLU(),
            torch.nn.Linear(50, 2))
    def forward(self, x):
        z = self.feature_ext(x)
        return z

#rescaler

class ScaledLayer(nn.Module):
    def __init__(self, l_dim = 2):
        super().__init__()
        self.l_dim = l_dim
        self.register_buffer("z_lower_bound", torch.zeros(l_dim))   #stored in the model but not trained by optimiser
        self.register_buffer("z_upper_bound", torch.ones(l_dim))    #https://discuss.pytorch.org/t/what-is-the-difference-between-register-buffer-and-register-parameter-of-nn-module/32723

    @torch.no_grad()                                                #no grad since not learnable parameters
    def set_bounds(self, z):
        self.z_lower_bound.copy_(z.min(dim = 0).values)               #update bounds
        self.z_upper_bound.copy_(z.max(dim = 0).values)

    def forward(self, z):
        if self.training:                                            #update bounds only during training, fixed for evaluation
            with torch.no_grad():                                    #we dont want graph created for these operations
                self.set_bounds(z)

        z_scaled = 2.0*((z - self.z_lower_bound)/((self.z_upper_bound - self.z_lower_bound).clamp_min(1e-4))) - 1.0
        z_scaled = z_scaled.clamp(-1.0,1.0)                      
        return z_scaled

#SVGP                                                                                             #https://docs.gpytorch.ai/en/stable/models.html
class SparseVariationalGPR(gpytorch.models.ApproximateGP):                                         #variational inference Hensman
    def __init__(self, inducing_z):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(           #https://docs.gpytorch.ai/en/v1.9.0/_modules/gpytorch/variational/cholesky_variational_distribution.html
            num_inducing_points = inducing_z.size(0))
        variational_strategy = gpytorch.variational.VariationalStrategy(                           #https://docs.gpytorch.ai/en/v1.6.0/variational.html
            self, inducing_z, variational_distribution, learn_inducing_locations = True)           
        super().__init__(variational_strategy)
        self.mean = gpytorch.means.ZeroMean()                                                    
        #self.covar_p = gpytorch.kernels.RBFKernel()                                               #RBF kernel
        self.covar = gpytorch.kernels.MaternKernel(nu = 2.5)                                     #matern kernel

    def forward(self, z):
        return gpytorch.distributions.MultivariateNormal(self.mean(z), self.covar(z))         

#combining all 3 together

class DKLnn(nn.Module):
    def __init__(self, d_input, n_inducing, device, dtype):
        super().__init__()
        self.feature_ext_layer = FNNfeatureExt(d_input = d_input).to(device = device, dtype = dtype)
        self.scale_layer = ScaledLayer(l_dim = 2).to(device = device, dtype = dtype)
        ind_points = torch.randn(n_inducing, 2, device = device, dtype = dtype)                                  #initialise z
        self.svgp_layer = SparseVariationalGPR(inducing_z = ind_points).to(device = device, dtype = dtype)

    def forward(self, x):
        z = self.feature_ext_layer(x)
        z = self.scale_layer(z)
        return self.svgp_layer(z)

#train the dkl model

def train_dkl(X, y, N, d, n_inducing, iters, lr, momentum, weight_decay):

    dkl_model = DKLnn(d_input = d, n_inducing = n_inducing, device = device, dtype = dtype)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device = device, dtype = dtype)
    dkl_model.train()
    likelihood.train()
    
    #full batch GD with momentum

    optimiser = torch.optim.SGD(list(dkl_model.parameters()) + list(likelihood.parameters()),
                                lr = lr, momentum = momentum, weight_decay = weight_decay)

    #torch.optim.lr_scheduler.ExponentialLR(optimiser, gamma = 0.998)      #tried decaying lr due to stability problems with large lr such as the paper

    elbo = gpytorch.mlls.VariationalELBO(likelihood, dkl_model.svgp_layer, num_data = N)  #https://docs.gpytorch.ai/en/latest/marginal_log_likelihoods.html
    for smth in range(iters):
        optimiser.zero_grad(set_to_none = True)
        train_pred = dkl_model(X)
        comp_loss = -elbo(train_pred, y)                                                        #negative ELBO as our loss function
        comp_loss.backward()
        #torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(likelihood.parameters()), max_norm = 30.0)        #tried gradient clipping for stability
                                                                   #https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html
        optimiser.step()

    dkl_model.eval()
    with torch.no_grad():                                                            #update and fix the scaling for predictions
        zs = dkl_model.feature_ext_layer(X)
        dkl_model.scale_layer.set_bounds(zs)

    return dkl_model, likelihood

#use the dkl model to estimate the continuation value

@torch.no_grad()                                                           #https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html
def posterior_mean(dkl_model, likelihood, X):
    dkl_model.eval()
    likelihood.eval()
    return dkl_model(X).mean


#the standard backward recursion algorithm

def dkl_maxcall_pricer(paths, N, d, K, T, n_dates, r,
                       n_iter = 1500, lr = 0.01, weight_decay = 1e-8, n_inducing = 200, momentum = 0.9):

    dt = T/n_dates

    v_cont = payoff_max_call(paths[:,n_dates,:], K = K)                   #value of the option at time T

    for i in range(n_dates - 1,0, -1):                                     #backwards recursion
        X_i = paths[:,i,:]
        y_i = v_cont
         
        #standardise for numerical stability 
        
        y_mean = y_i.mean()
        y_std = y_i.std().clamp_min(1e-6)    
        y_i_bar = (y_i - y_mean) / y_std

        dkl_model, likelihood = train_dkl(X = X_i, y = y_i_bar,
                                          N = N, d = d, n_inducing = n_inducing,
                                          iters = n_iter, lr = lr, momentum = momentum, weight_decay = weight_decay)

        cont_est = posterior_mean(dkl_model,likelihood,X_i) * y_std + y_mean                #estimated continuation value
        c_hat = torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype)) * cont_est     #estimated discounted continuation value

        g_i = payoff_max_call(X_i, K = K)                                   #payoff at time i
        exer = (g_i > c_hat) & (g_i > 0)                                    #exercise if payoff > estimated continuation value and payoff>0

        v_cont = torch.where(exer, g_i, torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype)) * v_cont)
                                                                            #update cashflows

    v0 = np.exp(-r*dt) * v_cont.mean().item()
    
    return v0

### Compute option value + computational time

In [ ]:
S0 = 110.0             #spot price
T = 3.0                #maturity
K = 100.0              #strike price
n_dates = 9            #num of exercise dates
r = 0.05               #risk free interest rate
q = 0.1                #divident yield
sigma = 0.2            #volatility
rho = 0.0              #correlation
N = 10000              #number of simulated paths
d = 50                 #number of underlyings

paths = simulate_gbm(S0 = S0, d = d, N = N, K = K, r = r, q = q, sigma = sigma, rho = rho,
                     T = T, n_dates = n_dates,
                     device = device, dtype = dtype)

start = time.time()

v = dkl_maxcall_pricer(paths = paths, N = N, d = d, K = K, T = T, n_dates = n_dates, r = r,
                       n_iter = 1500, lr = 0.01, weight_decay = 1e-8, n_inducing = 200, momentum = 0.9)
stop = time.time()


print( f"d = {d} | S0 = {S0} | {v:.6f} | time: {stop - start:.2f}s")